In [16]:
import re
from collections import defaultdict
from typing import List
from utils.exploit_gates1 import NetlistParser
import math


In [17]:
import re
from collections import defaultdict
from typing import List
from detection_codes.does_have_trojan3 import parse_verilog_signals, signals_to_dict
from utils.exploit_gates2 import NetlistParser
from utils.exploit_gates2 import transform_and_parse_with_originals







# Extract Trojan gates from the reference file
def extract_trojan_gates(filename):
    trojan_gates = []
    inside_block = False

    with open(filename, 'r') as file:
        for line in file:
            stripped = line.strip()
            if stripped == "TROJAN_GATES":
                inside_block = True
                continue
            if stripped == "END_TROJAN_GATES":
                break
            if inside_block:
                trojan_gates.append(stripped)

    return trojan_gates




class Signal:
    def __init__(self, name: str, size: int, sig_type: str):
        self.name = name
        self.size = size
        self.type = sig_type  # "None", "PI", or "PO"

    def __repr__(self):
        return f"Signal(name='{self.name}', size={self.size}, type='{self.type}')"


def strip_comments(text: str) -> str:
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
    text = re.sub(r"//.*?$", "", text, flags=re.M)
    return text


def parse_range_to_size(range_str: str) -> int:
    m = re.match(r"\[\s*(\d+)\s*:\s*(\d+)\s*\]", range_str) if range_str else None
    if not m:
        return 1
    a, b = int(m.group(1)), int(m.group(2))
    return abs(a - b) + 1


def extract_decl_names(decl_body: str):
    parts = [p.strip() for p in decl_body.split(",")]
    names = []
    for p in parts:
        base = re.split(r"\s*\[", p)[0].strip()
        base = re.split(r"\s*=", base)[0].strip()
        if base:
            names.append(base)
    return names


def parse_verilog_signals(text: str):
    clean = strip_comments(text)
    clean = re.sub(r"\s+", " ", clean)

    port_dir_map = {}
    for dir_kw in ["input", "output"]:
        pattern = rf"\b{dir_kw}\b\s+(?:reg\s+|wire\s+)?(?P<range>\[[^\]]+\]\s+)?(?P<names>[^;]+?)\s*;"
        for m in re.finditer(pattern, clean):
            rng = m.group("range")
            size = parse_range_to_size(rng) if rng else 1
            names = extract_decl_names(m.group("names"))
            for nm in names:
                if nm not in port_dir_map:
                    port_dir_map[nm] = (dir_kw, size)

    wire_info = {}
    for m in re.finditer(r"\bwire\b\s+(?P<range>\[[^\]]+\]\s+)?(?P<names>[^;]+?)\s*;", clean):
        rng = m.group("range")
        size = parse_range_to_size(rng) if rng else 1
        names = extract_decl_names(m.group("names"))
        for nm in names:
            prev = wire_info.get(nm)
            if prev is None or size > prev:
                wire_info[nm] = size

    all_names = set(wire_info.keys()) | set(port_dir_map.keys())
    signals = []
    for nm in sorted(all_names):
        if nm in port_dir_map:
            dir_kw, port_size = port_dir_map[nm]
            sig_type = "PI" if dir_kw == "input" else "PO"
            size = max(port_size, wire_info.get(nm, port_size))
        else:
            sig_type = "None"
            size = wire_info.get(nm, 1)
        signals.append(Signal(nm, size, sig_type))

    return signals

def signals_to_dict(signals):
    """
    Convert a list of Signal objects into a dictionary mapping
    signal name -> Signal object.
    """
    return {sig.name: sig for sig in signals}

In [18]:
Test_Design_Number = 28

In [19]:
target_file = f"./release_hidden_0923/release_hidden/design{Test_Design_Number}.v"

trojan_gates = []
trojan_gates_set = set()

new_parser = NetlistParser()
new_parser.parse_netlist(target_file)

with open(target_file, "r") as f:
    text = f.read()
signals = parse_verilog_signals(text)
signal_dict = signals_to_dict(signals)

def find_fanin_cone(gate_name: str) -> set:
    """
    Find the fanin cone of a gate until primary inputs or a dff using bfs.
    """
    gate = new_parser.get_gate(gate_name)
    fanin_inputs = set()
    queue = [gate]
    visited = set()
    visited.add(gate.name)
    while queue:
        current_gate = queue.pop(0)
        inputs = current_gate.inputs
        if current_gate.gate_type == 'dff':
            fanin_inputs.add(current_gate.output_net.split('[')[0])
            continue
        for input_name in inputs:
            driver_gate = new_parser.get_gate(input_name)
            if driver_gate is None:
                # print (input_name)
                if input_name[0] != '1':
                    if signal_dict[input_name.split('[')[0]].size > 1 and signal_dict[input_name.split('[')[0]].size <= 3:
                        fanin_inputs.add(input_name)
                    else:
                        fanin_inputs.add(input_name.split('[')[0])

            else:
                if driver_gate.name not in visited:
                    queue.append(driver_gate)
                    visited.add(driver_gate.name)
    return fanin_inputs

In [20]:
# Finding PrimaryWords
PrimaryWords = []
for bus_name in signal_dict:
    if signal_dict[bus_name].size == 1:
        continue
    if signal_dict[bus_name].type == "PI":
        PrimaryWords.append(bus_name)
        continue

for gate_name in new_parser.gates:
    gate = new_parser.get_gate(gate_name)
    if gate.gate_type == 'dff':
        dff_output_net = gate.output_net
        if '[' in dff_output_net:
            bus_name = dff_output_net.split('[')[0]
            if signal_dict[bus_name].size > 1:
                PrimaryWords.append(bus_name)

print("Length of PrimaryWords:", len(PrimaryWords))
print("PrimaryWords:", PrimaryWords)

Length of PrimaryWords: 4
PrimaryWords: ['n4', 'n5', 'n14', 'n8']


In [21]:
#print (new_parser.gates['g416'])

In [22]:
# Finding Candidate Outputs
CandidateOutputs = []
for gate_name in new_parser.gates:
    gate = new_parser.get_gate(gate_name)
    if gate.gate_type == 'dff':
        continue
    if gate.output_net in PrimaryWords:
        continue
    if '[' in gate.output_net:
        CandidateOutputs.append(gate_name)

In [23]:
# Step 1 - Per-gate cone analysis
W_PI_dict = defaultdict(set)
Bits_dict = defaultdict(lambda: defaultdict(list))
GateSet_dict = defaultdict(set)
depth_dict = defaultdict(int)
const_edge_dict = defaultdict(int)

for candidate_gate in CandidateOutputs:
    W_PI = set()
    Bits = defaultdict(list)
    GateSet = set()
    gate_count = 0
    depth = 0
    const_edge = 0
    # BFS - stopping at dffs and PrimaryWords and constants
    queue = [(candidate_gate, 0)]
    visited = set()
    bfs_gates_with_depth = set()
    visited.add(candidate_gate)
    bfs_gates_with_depth.add((candidate_gate, 0))
    while queue:
        current_gate_name, current_depth = queue.pop(0)
        gate_count += 1
        current_gate = new_parser.get_gate(current_gate_name)
        if current_gate.gate_type == 'dff':
            continue
        inputs = current_gate.inputs
        for input_name in inputs:
            if input_name not in new_parser.gates:
                continue
            driver_gate = new_parser.get_gate(input_name)
            if driver_gate is None:
                continue
            if driver_gate.gate_type == 'dff':
                continue
            output_net = driver_gate.output_net
            if output_net in PrimaryWords:
                continue
            if '[' in output_net:
                continue
            if driver_gate.name not in visited:
                queue.append((driver_gate.name, current_depth + 1))
                visited.add(driver_gate.name)
                bfs_gates_with_depth.add((driver_gate.name, current_depth + 1))
    
    for gate_name, gate_depth in bfs_gates_with_depth:
        GateSet.add(gate_name)
        gate_count += 1
        for input_name in new_parser.get_gate(gate_name).inputs:
            if input_name not in new_parser.gates:
                index = -1
                if input_name[0] == '1':
                    const_edge += 1
                    continue
                if '[' in input_name:
                    index = input_name.split('[')[1].split(']')[0]
                    input_name = input_name.split('[')[0]
                if input_name in PrimaryWords:
                    W_PI.add(input_name)
                    if input_name not in Bits:
                        Bits[input_name] = []
                    Bits[input_name].append(int(index))
                continue
            driver_gate = new_parser.get_gate(input_name)
            if driver_gate is None:
                continue
            input_net = driver_gate.output_net
            if '[' in input_net:
                index = input_net.split('[')[1].split(']')[0]
                input_net = input_net.split('[')[0]
            if input_net in PrimaryWords:
                W_PI.add(input_net)
                if input_net not in Bits:
                    Bits[input_net] = []
                Bits[input_net].append(int(index))
                continue
        
        depth = max(depth, gate_depth)

    num_PI_words = len(W_PI)
    total_leaf_bits = 0
    for word in Bits:
        total_leaf_bits += len(Bits[word])
    min_bits_per_word = min([len(Bits[word]) for word in Bits]) if len(Bits) > 0 else 0
    W_PI_dict[candidate_gate] = W_PI
    Bits_dict[candidate_gate] = Bits
    GateSet_dict[candidate_gate] = GateSet
    depth_dict[candidate_gate] = depth
    const_edge_dict[candidate_gate] = const_edge


In [24]:
# Step 2 - Group candidates by the base of their output net
CandidateGroups = defaultdict(list)
for gate_name in CandidateOutputs:
    gate = new_parser.get_gate(gate_name)
    base_output_net = gate.output_net.split('[')[0]
    CandidateGroups[base_output_net].append(gate_name)

IntersectPI_dict = defaultdict(set)
UnionPI_dict = defaultdict(set)


T89_like_blocks = set()

for base_net in CandidateGroups:
    width = len(CandidateGroups[base_net])
    # IntersectPI = ⋂_{g∈C} W_PI(g)
    IntersectPI = None
    for gate_name in CandidateGroups[base_net]:
        W_PI = W_PI_dict[gate_name]
        if IntersectPI is None:
            IntersectPI = W_PI
        else:
            IntersectPI = IntersectPI.intersection(W_PI)
    # UnionPI = ⋃_{g∈C} W_PI(g)
    UnionPI = None
    for gate_name in CandidateGroups[base_net]:
        W_PI = W_PI_dict[gate_name]
        if UnionPI is None:
            UnionPI = W_PI
        else:
            UnionPI = UnionPI.union(W_PI)
    # avg_gate = mean(gate_count(g))
    avg_gate = 0
    for gate_name in CandidateGroups[base_net]:
        avg_gate += len(GateSet_dict[gate_name])
    avg_gate = avg_gate / width
    # var_gate = var(gate_count(g))
    var_gate = 0
    for gate_name in CandidateGroups[base_net]:
        var_gate += (len(GateSet_dict[gate_name]) - avg_gate) ** 2
    var_gate = var_gate / width
    flag = False
    for w in IntersectPI:
        # avg_bits_w = mean_g |Bits[g][w]|
        avg_bits_w = 0
        for gate_name in CandidateGroups[base_net]:
            Bits = Bits_dict[gate_name]
            avg_bits_w += len(Bits[w]) if w in Bits else 0
        avg_bits_w = avg_bits_w / width
        

        # span_w = max_g(max(Bits[g][w])) − min_g(min(Bits[g][w]))
        span_w = -1
        min_bit = None
        max_bit = None
        for gate_name in CandidateGroups[base_net]:
            Bits = Bits_dict[gate_name]
            if w in Bits:
                bits_list = Bits[w]
                local_min = min([b for b in bits_list if b is not None])
                local_max = max([b for b in bits_list if b is not None])
                if min_bit is None or local_min < min_bit:
                    min_bit = local_min
                if max_bit is None or local_max > max_bit:
                    max_bit = local_max
        if min_bit is not None and max_bit is not None:
            # print (max_bit, min_bit)
            span_w = max_bit - min_bit
        if avg_bits_w < 1:
            flag = True
        if span_w < 0: # adjustable threshold (what should this be?)
            flag = True
        
    IntersectPI_dict[base_net] = IntersectPI
    UnionPI_dict[base_net] = UnionPI

    # if len(CandidateGroups[base_net]) < 4:
    #     print (base_net, len(CandidateGroups[base_net]))
    #     continue
    std_gate = math.sqrt(var_gate)

    cv_gate = std_gate / avg_gate if avg_gate > 0 else 1.0
    # print (cv_gate)
    if cv_gate > 10:   # 20% variation
        continue
    if len(IntersectPI) < 0:
        continue
    if flag:
        print (base_net,"flagged")
        continue
    if avg_gate < 0: #adjustable threshold (what should this be?)
        continue
    T89_like_blocks.add(base_net)




    


In [25]:
for block in T89_like_blocks:
    for gate_name in CandidateGroups[block]:
        if gate_name in trojan_gates:
            continue
        trojan_gates.append(gate_name)
        for g in GateSet_dict[gate_name]:
            if g in trojan_gates:
                continue
            trojan_gates.append(g)

print (len(trojan_gates))
print (trojan_gates)
print (T89_like_blocks)

64
['g31', 'g43', 'g49', 'g56', 'g58', 'g59', 'g64', 'g33', 'g46', 'g9', 'g35', 'g4', 'g17', 'g67', 'g21', 'g32', 'g34', 'g22', 'g0', 'g71', 'g18', 'g45', 'g60', 'g8', 'g72', 'g69', 'g24', 'g38', 'g66', 'g55', 'g12', 'g25', 'g14', 'g39', 'g6', 'g57', 'g28', 'g7', 'g37', 'g15', 'g3', 'g2', 'g26', 'g10', 'g19', 'g47', 'g51', 'g53', 'g73', 'g11', 'g42', 'g61', 'g36', 'g63', 'g13', 'g44', 'g16', 'g41', 'g50', 'g65', 'g23', 'g27', 'g68', 'g70']
{'n9', 'n10', 'n13', 'n8', 'n11', 'n12'}


In [26]:
print (trojan_gates)

['g31', 'g43', 'g49', 'g56', 'g58', 'g59', 'g64', 'g33', 'g46', 'g9', 'g35', 'g4', 'g17', 'g67', 'g21', 'g32', 'g34', 'g22', 'g0', 'g71', 'g18', 'g45', 'g60', 'g8', 'g72', 'g69', 'g24', 'g38', 'g66', 'g55', 'g12', 'g25', 'g14', 'g39', 'g6', 'g57', 'g28', 'g7', 'g37', 'g15', 'g3', 'g2', 'g26', 'g10', 'g19', 'g47', 'g51', 'g53', 'g73', 'g11', 'g42', 'g61', 'g36', 'g63', 'g13', 'g44', 'g16', 'g41', 'g50', 'g65', 'g23', 'g27', 'g68', 'g70']


In [27]:


parser = NetlistParser()
parser.parse_netlist(target_file)
reference_Trojans_file = f"./release_hidden_0923/release_hidden/result{Test_Design_Number}.txt"
actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
score = 2
# print(f"Number of trojan gates in reference file: {len(actual_trojan_gates)}")

# print (f"Actual Trojan gates: {sorted(actual_trojan_gates, key=lambda x: (len(x), x), reverse=True)}")
# print (f"Predicted Trojan gates: {sorted(trojan_gates, key=lambda x: (len(x), x), reverse=True)}")


true_positive = 0
for gate in parser.gates:
    if gate in trojan_gates and gate in actual_trojan_gates:
        true_positive = true_positive + 1


#false_positive = len(trojan_gates - set(actual_trojan_gates))
false_positive = 0
for gate in trojan_gates:
    if gate not in actual_trojan_gates:
        false_positive = false_positive + 1

#false_negative = len(set(actual_trojan_gates) - trojan_gates)
false_negative = 0
for gate in actual_trojan_gates:
    if gate not in trojan_gates:
        false_negative = false_negative + 1

true_negative = 0
for gate in parser.gates:
    if gate not in trojan_gates and gate not in actual_trojan_gates:
        true_negative = true_negative + 1


TPR = true_positive / len(actual_trojan_gates) if actual_trojan_gates else 0
FPR = false_positive / (len(actual_trojan_gates) + false_negative) if (len(actual_trojan_gates) + false_negative) > 0 else 0
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
f1_score = 2 * (recall * precision) / (recall + precision) if (recall + precision) > 0 else 0

score = score + f1_score

# print(f"True Positive Rate (TPR): {TPR:.4f}")
# print(f"False Positive Rate (FPR): {FPR:.4f}")
# print(f"True Positive Count: {true_positive}")
# print(f"False Positive Count: {false_positive}")
# print(f"False Negative Count: {false_negative}")
# print(f"True Negative Count: {true_negative}")
# print (f"precision: {precision:.4f}")
# print (f"recall: {TPR:.4f}")
# print (f"F1 score: {f1_score:.4f}")
# print (f"total number of gates: {len(parser.gates)}")
# print_excel_outputs = [true_positive, false_positive, false_negative, true_negative, precision, TPR, f1_score]
# print('\t'.join(map(str, print_excel_outputs)))
# print(Test_Design_Number)

# for gate_name in actual_trojan_gates:
#     if gate_name not in trojan_gates:
#         print(parser.get_gate(gate_name))
print_excel_outputs = [Test_Design_Number,true_positive, false_positive, false_negative, true_negative, precision, TPR, f1_score, score]
print('\t'.join(map(str, print_excel_outputs)))

28	38	26	0	10	0.59375	1.0	0.7450980392156863	2.7450980392156863


In [28]:
for gate_name in trojan_gates:
    print(gate_name, parser.get_gate(gate_name).output_net)

g31 n9[1]
g43 n9[2]
g49 n9[6]
g56 n9[0]
g58 n9[4]
g59 n9[5]
g64 n9[3]
g33 n10[1]
g46 n28
g9 n42
g35 n46
g4 n41
g17 n35
g67 n30
g21 n31
g32 n36
g34 n10[2]
g22 n49
g0 n51
g71 n27
g18 n39
g45 n54
g60 n43
g8 n52
g72 n29
g69 n47
g24 n38
g38 n10[0]
g66 n26
g55 n10[3]
g12 n40
g25 n53
g14 n55
g39 n44
g6 n37
g57 n32
g28 n50
g7 n13[4]
g37 n45
g15 n33
g3 n56
g2 n48
g26 n57
g10 n8[0]
g19 n8[3]
g47 n8[2]
g51 n8[1]
g53 n8[5]
g73 n8[4]
g11 n11[0]
g42 n21
g61 n22
g36 n24
g63 n23
g13 n11[1]
g44 n25
g16 n58
g41 n34
g50 n11[3]
g65 n11[2]
g23 n12[2]
g27 n12[1]
g68 n12[0]
g70 n12[3]
